
# LightGBM vs XGBoost 

In [ ]:

# Import Required Libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


In [ ]:

# Load Dataset

train_df = pd.read_csv("Titanic_train(1).csv")
test_df = pd.read_csv("Titanic_test(1).csv")

print("Training Dataset Shape:", train_df.shape)
print("Testing Dataset Shape:", test_df.shape)

train_df.head()


In [ ]:

# Dataset Information

print(train_df.info())

print("\nSummary Statistics")
print(train_df.describe())



# Exploratory Data Analysis (EDA)


In [ ]:

# Check Missing Values

print(train_df.isnull().sum())


In [ ]:

# Histograms

train_df.hist(figsize=(15,10), bins=20)

plt.tight_layout()

plt.show()


In [ ]:

# Boxplots

plt.figure(figsize=(12,6))

sns.boxplot(data=train_df.select_dtypes(include=np.number))

plt.xticks(rotation=90)

plt.show()


In [ ]:

# Survival Count Plot

sns.countplot(x='Survived', data=train_df)

plt.title("Survival Distribution")

plt.show()


In [ ]:

# Survival by Sex

sns.barplot(x='Sex', y='Survived', data=train_df)

plt.title("Survival by Gender")

plt.show()


In [ ]:

# Correlation Heatmap

plt.figure(figsize=(10,8))

sns.heatmap(train_df.corr(numeric_only=True), annot=True, cmap='coolwarm')

plt.title("Correlation Heatmap")

plt.show()



# Data Preprocessing


In [ ]:

# Handle Missing Values

train_df['Age'] = train_df['Age'].fillna(train_df['Age'].median())
test_df['Age'] = test_df['Age'].fillna(test_df['Age'].median())

train_df['Embarked'] = train_df['Embarked'].fillna(train_df['Embarked'].mode()[0])
test_df['Embarked'] = test_df['Embarked'].mode()[0]

test_df['Fare'] = test_df['Fare'].fillna(test_df['Fare'].median())

# Drop unnecessary columns
drop_cols = ['PassengerId', 'Name', 'Ticket', 'Cabin']

train_df.drop(columns=drop_cols, inplace=True)
test_df.drop(columns=drop_cols, inplace=True)

train_df.head()


In [ ]:

# Encode Categorical Variables

label_encoder = LabelEncoder()

for col in ['Sex', 'Embarked']:
    train_df[col] = label_encoder.fit_transform(train_df[col])
    test_df[col] = label_encoder.transform(test_df[col])

train_df.head()


In [ ]:

# Features and Target Variable

X = train_df.drop('Survived', axis=1)
y = train_df['Survived']


In [ ]:

# Split Dataset

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("Training Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)



# LightGBM Model


In [ ]:

# Build LightGBM Model

lgbm_model = LGBMClassifier(random_state=42)

lgbm_model.fit(X_train, y_train)

# Predictions
lgbm_pred = lgbm_model.predict(X_test)


In [ ]:

# LightGBM Evaluation

lgbm_accuracy = accuracy_score(y_test, lgbm_pred)
lgbm_precision = precision_score(y_test, lgbm_pred)
lgbm_recall = recall_score(y_test, lgbm_pred)
lgbm_f1 = f1_score(y_test, lgbm_pred)

print("LightGBM Accuracy:", lgbm_accuracy)
print("LightGBM Precision:", lgbm_precision)
print("LightGBM Recall:", lgbm_recall)
print("LightGBM F1 Score:", lgbm_f1)

print("\nClassification Report")
print(classification_report(y_test, lgbm_pred))



# XGBoost Model


In [ ]:

# Build XGBoost Model

xgb_model = XGBClassifier(
    random_state=42,
    eval_metric='logloss'
)

xgb_model.fit(X_train, y_train)

# Predictions
xgb_pred = xgb_model.predict(X_test)


In [ ]:

# XGBoost Evaluation

xgb_accuracy = accuracy_score(y_test, xgb_pred)
xgb_precision = precision_score(y_test, xgb_pred)
xgb_recall = recall_score(y_test, xgb_pred)
xgb_f1 = f1_score(y_test, xgb_pred)

print("XGBoost Accuracy:", xgb_accuracy)
print("XGBoost Precision:", xgb_precision)
print("XGBoost Recall:", xgb_recall)
print("XGBoost F1 Score:", xgb_f1)

print("\nClassification Report")
print(classification_report(y_test, xgb_pred))



# Cross Validation


In [ ]:

# Cross Validation Scores

lgbm_cv = cross_val_score(
    lgbm_model,
    X,
    y,
    cv=5,
    scoring='accuracy'
)

xgb_cv = cross_val_score(
    xgb_model,
    X,
    y,
    cv=5,
    scoring='accuracy'
)

print("LightGBM CV Accuracy:", lgbm_cv.mean())
print("XGBoost CV Accuracy:", xgb_cv.mean())



# Hyperparameter Tuning


In [ ]:

# Hyperparameter Tuning for LightGBM

param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [3, 5],
    'learning_rate': [0.01, 0.1]
}

grid_search = GridSearchCV(
    LGBMClassifier(random_state=42),
    param_grid,
    cv=3,
    scoring='accuracy'
)

grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)

print("Best CV Score:", grid_search.best_score_)


In [ ]:

# Confusion Matrix - LightGBM

cm_lgbm = confusion_matrix(y_test, lgbm_pred)

sns.heatmap(cm_lgbm, annot=True, fmt='d', cmap='Blues')

plt.title("LightGBM Confusion Matrix")

plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()


In [ ]:

# Confusion Matrix - XGBoost

cm_xgb = confusion_matrix(y_test, xgb_pred)

sns.heatmap(cm_xgb, annot=True, fmt='d', cmap='Greens')

plt.title("XGBoost Confusion Matrix")

plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()



# Comparative Analysis


In [ ]:

# Compare Model Performance

comparison = pd.DataFrame({
    'Model': ['LightGBM', 'XGBoost'],
    'Accuracy': [lgbm_accuracy, xgb_accuracy],
    'Precision': [lgbm_precision, xgb_precision],
    'Recall': [lgbm_recall, xgb_recall],
    'F1 Score': [lgbm_f1, xgb_f1]
})

comparison



## Observations

### LightGBM
- Faster training speed.
- Efficient with large datasets.
- Lower memory usage.

### XGBoost
- Often achieves slightly higher accuracy.
- Better handling of complex patterns.
- More computationally intensive.

Both models performed well on the Titanic dataset.


# Additional Interpretation and Visualization

In [ ]:

# Feature Importance - LightGBM

import pandas as pd
import matplotlib.pyplot as plt

feature_imp = pd.DataFrame({
    'Feature': X.columns,
    'Importance': lgbm_model.feature_importances_
}).sort_values('Importance', ascending=False)

print(feature_imp.head())

feature_imp.head(10).plot(
    x='Feature',
    y='Importance',
    kind='bar',
    figsize=(8,4),
    legend=False
)

plt.title('Top 10 Important Features - LightGBM')
plt.show()


In [ ]:

# Hyperparameter Tuning Results Visualization

comparison.plot(
    x='Model',
    y='Accuracy',
    kind='bar',
    legend=False,
    figsize=(6,4)
)

plt.ylabel('Accuracy')
plt.title('Model Accuracy Comparison')
plt.show()
